# Chapter 7: Search In-Depth
## Topic 1: Sparse Retrieval — TF-IDF → BM25

**One notebook. Theory + Code together.**
- Part A: Theory (Concept -> Revision Summary)
- Part B: Code (7 modules, run top to bottom)


## Part A: Theory

### 1. Concept, Intuition, and Why It Exists

- Retrieval means: given a query, find the most relevant documents/chunks from a big pile.
- **Sparse retrieval** finds relevant documents by matching **words**, not meaning.
- "Sparse" means: if you turn a document into a vector where each position = one word in the vocabulary, almost all positions are 0, because any single document only uses a tiny fraction of all possible words.
- Example: your email corpus has thousands of unique words. One email averages ~31 words. So a vector representing that email is mostly zeros — hence "sparse."
- This is the opposite of **dense retrieval** (Topic 2), where every position in the vector has a non-zero value (like embeddings).

**Why this exists:**

- Before deep learning, this is literally how search engines worked — count words, weight them smartly, rank documents.
- It's still used today in every production RAG system alongside dense retrieval, because it solves problems dense retrieval cannot.
- It's a prerequisite for Hybrid Search (Topic 4), Reranking (Topic 7), and the entire retrieval layer of RAG.

**Connect it to what you already built:**

- Your Chapter 1 `classify_by_keyword_rules()` function is the crudest possible version of sparse retrieval.
- It just checks: does this word appear in the email — yes or no?
- No counting how many times, no idea if the word is rare or common, no ranking.
- BM25 is the "grown-up" version of this same idea.


### 2. Internal Working — Step by Step

**Step 1 — TF-IDF, the first real attempt**

- A word is important to a document if:
  - It appears a lot **in that document** (Term Frequency — TF)
  - It does NOT appear in many other documents (Inverse Document Frequency — IDF)
- Multiply the two together -> TF-IDF score.

```text
TF(word, doc)   = how many times word appears in doc / total words in doc
IDF(word)       = log( total documents / documents containing word )
TF-IDF          = TF x IDF
```

- Word "hai" appears in almost every email -> very common -> IDF is tiny -> score is low.
- Word "byaj" (interest, in Hinglish) appears in very few emails -> rare -> IDF is high -> score is high. This is exactly why TF-IDF automatically suppresses common junk words without needing a manual stop-word list.

**Real numbers from your own project (Chapter 0):**

- "gaya" -> appears 2 times, in 613 of 2,500 emails -> TF-IDF = 0.750 (high, because repeated)
- "ho" -> appears 1 time, in 598 of 2,500 emails -> TF-IDF = 0.379 (rare -> boosted)
- "loan" -> appears 1 time, in 642 of 2,500 emails -> TF-IDF = 0.368 (topically useful)
- "is" -> appears 1 time, in 1,053 of 2,500 emails -> TF-IDF = 0.291 (common -> downweighted)
- "hai" -> appears 1 time, in 1,190 of 2,500 emails -> TF-IDF = 0.272 (most common -> lowest weight)

**Step 2 — Where TF-IDF breaks**

- **Repetition problem (no saturation):** If a word appears 40 times, TF-IDF multiplies its score by roughly 40. But a word appearing 40 times doesn't really make a document 40x more relevant. Your `03_FD_SOPs.pdf` repeats "premature withdrawal" across many steps, so TF-IDF ranks this SOP above a short, clean FAQ answer — not because it's more relevant, but because it repeats words more.
- **Length problem (no proper normalization):** A long 10-page policy document and a 1-line FAQ both mention "interest." TF-IDF divides by total words in the doc, but this doesn't fully fix the bias — long documents still tend to win purely because they're long, not because they're more relevant.

**Step 3 — BM25, the fix that became the industry standard**

- BM25 = "Best Match 25" — the 25th version of a search-ranking formula from the 1980s-90s (Okapi project, City University London).
- It's the default search algorithm inside Elasticsearch, OpenSearch, and Solr today.
- It fixes both TF-IDF problems:
  - **Fixes repetition** using a "diminishing returns" curve controlled by a parameter called `k1`. Going from 1 mention to 2 mentions of a word adds a good chunk of score. Going from 39 to 40 mentions adds almost nothing. Like adding sugar to tea — the first spoon changes the taste a lot, the tenth spoon barely matters.
  - **Fixes length bias** using a parameter called `b`. It compares a document's length to the average document length in the whole corpus, and pulls down scores for documents that are much longer than average.

```text
BM25(doc, query) = sum over each query word of:
   IDF(word) x [ TF(word, doc) x (k1 + 1) ] / [ TF(word, doc) + k1 x (1 - b + b x doc_length/avg_doc_length) ]
```

- `k1` (usually 1.2): controls how fast repetition stops mattering.
- `b` (usually 0.75): controls how much document length is penalized (0 = ignore length completely, 1 = fully normalize by length).
- With k1=1.2: going from TF=1 to TF=40 (a 40x increase in raw repetition) only takes the saturation curve from 1.0 to about 2.14 — roughly 2x, not 40x. That gap between "40x more repetition" and "about 2x more score" is saturation working correctly.


### 3. How It Is Implemented in This Project

- The project's knowledge base (policy/FAQ/SOP/product chunks derived from the 4 source PDFs, plus `data/fd_master_database.csv`) are the "documents" BM25 searches over.
- In code, we use the `rank_bm25` Python library's `BM25Okapi` class:
  - Tokenize every chunk (lowercase + split into words).
  - Build the BM25 index by passing all tokenized chunks to `BM25Okapi(...)`.
  - Call `.get_scores(query_tokens)` to get a ranked score for every chunk against a query.
- Compare this against `classify_by_keyword_rules()` from Chapter 1 — that function only says yes/no, BM25 gives an actual ranked score.
- Later, in Chapter 7 Topic 4 (Hybrid Search), BM25's ranked list gets combined with the dense embedding ranked list using Reciprocal Rank Fusion (RRF).
- The code below (Part B) implements this step by step in 7 runnable modules.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Vocabulary mismatch (the biggest issue for this project):** 64% of your emails are Hinglish ("byaj," "machurity"), but the policy documents are in English. If none of the query's words appear in a document, BM25 gives that document a score of **exactly zero** — no partial credit. This is BM25's single biggest weakness for this dataset.
- **False keyword overlap:** 145 Non-FD emails in the corpus contain FD-related words purely by coincidence, which can confuse both the rule-based classifier and BM25 ranking.
- **Small corpus IDF instability:** With only ~17 knowledge base pages, "rare" words aren't statistically reliable.
- **Scores are not stable over time:** Adding a new document shifts IDF for every word it contains, which changes scores for every existing document containing those words. Never hardcode a score threshold like "relevant if score > 5."
- **Debugging tip:** If BM25 returns all zeros for a query, check whether the query and document tokenization actually share any words — almost always a vocabulary mismatch, not a bug.
- **Monitoring in production:** Track Recall@K on a labeled evaluation set over time, and re-check it whenever the corpus grows significantly.
- **Scaling / latency:** `rank_bm25` scans every document for every query (O(n) per query). Production search engines use an **inverted index** — a lookup table from word to list of documents containing it — so a query only touches documents that contain at least one query word. At 500K chunks, sharding the index across multiple machines lets each shard search in parallel, with a coordinator merging results.
- **Cost:** BM25 is essentially free to run — no GPU, no model, no API calls, pure CPU math.
- **Security:** Minimal injection risk compared to prompt-based systems, but query inputs should still be sanitized before tokenization.
- **Deployment:** For small corpora, an in-memory `rank_bm25` index rebuilt on startup is fine. For production scale, use a dedicated search engine (Elasticsearch/OpenSearch) with a persistent inverted index supporting incremental updates.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Why BM25 instead of raw TF-IDF:** BM25 fixes the repetition and length bias problems, at the cost of two extra parameters (`k1`, `b`) that need occasional tuning.
- **Why default k1=1.2, b=0.75:** Reasonable defaults for general text, but repetition-heavy domains might want a higher k1, and very short documents (like your ~31-word emails) might need `b` tuned lower.
- **The real dilemma — BM25 alone vs BM25 + Dense (Hybrid):** BM25 is fast, cheap, and exact, but blind to meaning. Dense retrieval understands meaning but can miss exact codes/IDs and costs more. In real production RAG, you almost always run both and merge results with something like RRF.

### 6. Alternatives and When to Use Each

- **Binary keyword match (Chapter 1 style):** Use only for very simple, high-precision rule-based classification tasks — not for ranked search.
- **TF-IDF:** Fine for quick prototypes or very small corpora where BM25's improvements barely matter.
- **BM25:** Default choice for production keyword-based search — use whenever exact term matching matters (codes, IDs, rare product names).
- **Dense retrieval:** Use when queries and documents use different words for the same meaning (paraphrased questions, Hinglish vs English).
- **Hybrid (BM25 + Dense):** Use in real production RAG — this project's actual direction.

### 7. Common Mistakes and Production Failures

- Treating BM25 scores as absolute truth (e.g., hardcoding "score > 5 = relevant").
- Assuming BM25 "should" find semantically related documents when it mathematically cannot if there's zero word overlap.
- Not removing/handling stop words at index time — wastes index space and lookup time even though BM25's math suppresses them somewhat.
- Applying English stemmers directly on Hinglish text — this can corrupt transliterated Hindi words rather than help.
- Replacing BM25 entirely with dense retrieval assuming "transformers are always better" — ignoring BM25's reliability for exact codes and brand-new terms with no training signal.


### 8. Lead-Level Interview Questions

**Basic**

- Q: Why does BM25 return a score of zero for some documents?
  A: BM25 only measures word overlap. If none of the query's words appear in a document, there's nothing to score, regardless of semantic relevance. This is the vocabulary mismatch problem.

- Q: What does the `k1` parameter control?
  A: How quickly repeated word occurrences stop adding to the score. Higher k1 = repetition matters more; lower k1 = closer to binary present/absent.

**Intermediate**

- Q: Why do BM25 scores change when you add new documents to the corpus?
  A: Adding documents changes document frequency for every word they contain, which changes IDF, which changes scores for every document containing those words. Never hardcode absolute score thresholds.

- Q: Given a 2,500-email Hinglish corpus and English policy documents, what BM25 failure modes would you expect?
  A: Vocabulary mismatch, small-corpus IDF instability, and less meaningful length normalization since emails are short and similarly sized.

**Advanced**

- Q: A teammate wants to fully replace BM25 with dense retrieval. How do you respond?
  A: Dense retrieval is better at meaning, but BM25 is more reliable for exact matches. The right production answer is usually both in parallel, merged with something like Reciprocal Rank Fusion.

- Q: Your BM25 Recall@10 is 0.61. What do you do before jumping to dense retrieval?
  A: Diagnose why it's missing results first. Fix what's fixable within BM25 (spelling variants, query expansion). Only genuinely meaning-based misses justify adding dense retrieval.

**Scenario-based**

- Q: BM25 search over 500K chunks takes 8ms per query, but you need 5,000 queries per second. How do you scale?
  A: Shard the corpus across multiple machines using an inverted index, each shard searches in parallel, a coordinator merges results.

**Follow-up questions to expect:**

- "What happens to latency if you increase corpus size 10x without sharding?"
- "How would you detect that vocabulary mismatch, not a bug, is causing zero scores?"
- "If you could only pick one of BM25 or dense retrieval, which would you pick?" (Trick question — neither alone is sufficient.)


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **Inverted Index:** The real reason BM25 is fast in production — maps each word to the list of documents containing it. `rank_bm25` does NOT do this — it scans everything, fine for small demos but too slow at real scale.
- **Where BM25 mathematically comes from:** A practical approximation of a probability model (Robertson-Sparck Jones, 1976) estimating "how likely is this document relevant, given this query."
- **Stemming:** Reducing words to their root form can help BM25 match more variants, but English stemming rules on Hinglish text can break words rather than help.
- **Stop words:** BM25 naturally downweights common filler words through IDF, but stripping obvious stop words before indexing still saves space and speed.

### 10. Quick Revision Summary (for last-minute interview prep)

> Sparse retrieval matches on words, not meaning. Chapter 1's rule-based classifier was pure binary sparse retrieval. TF-IDF improved on that by weighting words based on how rare they are across the corpus, but has two flaws: repeated words add score without limit, and long documents unfairly get boosted. BM25 fixes both flaws — `k1` makes repeated word occurrences add diminishing returns, and `b` properly normalizes score by comparing document length to the corpus average. This made BM25 the industry-standard keyword search algorithm — fast, cheap, and exact-match reliable. Its one big blind spot: if a query and a document share zero words, BM25 scores it exactly zero, even if they mean the same thing. This is the vocabulary mismatch problem — which is exactly why Dense Retrieval (Topic 2) and Hybrid Search (Topic 4) exist.


## Part B: Code (Modularized)

Run cells top to bottom. Each module is self-contained after Module 1 has been run once.

- Module 1: Setup — shared knowledge base (run this first)
- Module 2: Binary keyword search (what Chapter 1's rule engine does)
- Module 3: TF-IDF retrieval — and where it breaks
- Module 4: BM25 retrieval — the two fixes (k1 saturation + b length normalization)
- Module 5: k1 saturation — seeing the diminishing-returns curve
- Module 6: IDF comparison — TF-IDF vs BM25 side by side
- Module 7: Hinglish vocabulary mismatch — the one thing BM25 cannot fix


### Module 1: Setup — Shared Knowledge Base

Run this first. Defines `KNOWLEDGE_BASE`, `CORPUS_TEXTS`, `LABELS`, `QUERY`, and `tokenize()` used by every module below.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Setup and Shared Knowledge Base
#
# This builds a tiny 6-chunk knowledge base that stands in for the
# 4 real source documents in the project:
#   01_FD_FAQ.pdf, 02_FD_Product_Guide.pdf,
#   03_FD_SOPs.pdf, 04_FD_Policy_Reference.pdf
#
# On purpose it includes:
#   - Doc 0: a short, clean FAQ answer -> should rank HIGH for penalty queries
#   - Doc 2: a long, repetitive SOP -> shows the TF-IDF repetition problem
#   - Doc 5: a doc using different words for the same idea -> shows the
#            vocabulary mismatch problem BM25 cannot fix
# ------------------------------------------------------------------

import math
import numpy as np
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine

# each dict below is one "chunk" of the knowledge base
KNOWLEDGE_BASE = [
    {
        "id": 0,
        "text": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",
        "source": "01_FD_FAQ.pdf",
        "doc_type": "faq",
    },
    {
        "id": 1,
        "text": "Fixed Deposit premature closure is allowed subject to applicable penalty charges as per RBI guidelines.",
        "source": "04_FD_Policy_Reference.pdf",
        "doc_type": "policy",
    },
    {
        "id": 2,
        # this chunk repeats "premature withdrawal" on purpose -- it will
        # show us the TF-IDF repetition problem later in Module 3
        "text": (
            "This SOP covers Fixed Deposit premature withdrawal procedures. "
            "Step 1: Verify FD account for premature withdrawal eligibility. "
            "Step 2: Check FD premature withdrawal penalty applicable rate. "
            "Step 3: Calculate premature withdrawal penalty on FD interest rate. "
            "Step 4: Process premature withdrawal and apply 1 percent penalty. "
            "Step 5: Credit FD amount after premature withdrawal penalty deduction. "
            "Step 6: Generate receipt for FD premature withdrawal with penalty details. "
            "Step 7: Update FD records after premature withdrawal completion."
        ),
        "source": "03_FD_SOPs.pdf",
        "doc_type": "sop",
    },
    {
        "id": 3,
        "text": "The Fixed Deposit interest rate for 24 months is 7.1 percent per annum.",
        "source": "02_FD_Product_Guide.pdf",
        "doc_type": "product",
    },
    {
        "id": 4,
        "text": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.",
        "source": "02_FD_Product_Guide.pdf",
        "doc_type": "product",
    },
    {
        "id": 5,
        # same meaning as doc 0 and 1, but uses "deposit account" and "returns"
        # instead of "FD" and "interest" -- this is our vocabulary mismatch example
        "text": "Early exit from your deposit account will attract a deduction from your returns.",
        "source": "04_FD_Policy_Reference.pdf",
        "doc_type": "policy",
    },
]

# pull out just the text strings -- most retrieval functions want a plain list
CORPUS_TEXTS = [doc["text"] for doc in KNOWLEDGE_BASE]

# a readable label per doc, used only for printing results nicely
LABELS = [f"[{doc['doc_type'].upper():<7}] {doc['source']}" for doc in KNOWLEDGE_BASE]

# the query we will reuse across most modules below
QUERY = "premature withdrawal penalty FD"


def tokenize(text: str) -> list:
    """
    Turns a sentence into a list of lowercase words.
    Example: "FD Penalty" -> ["fd", "penalty"]

    NOTE: in a real production pipeline for this project, you would also:
      - normalise Hinglish spellings (byaj -> interest, machurity -> maturity)
      - apply stemming carefully (English stemmers can break Hinglish words)
    We skip that here to keep the demo simple and focused on BM25 itself.
    """
    return text.lower().split()


# ------------------------------------------------------------------
# Just print a summary so we can see what we loaded
# ------------------------------------------------------------------
print("=" * 65)
print("KNOWLEDGE BASE LOADED")
print("=" * 65)
for doc in KNOWLEDGE_BASE:
    words = len(doc["text"].split())
    print(f"  Doc {doc['id']} | {words:>3} words | [{doc['doc_type'].upper():<7}] {doc['source']}")
    print(f"         {doc['text'][:65]}...")

avg_len = sum(len(d["text"].split()) for d in KNOWLEDGE_BASE) / len(KNOWLEDGE_BASE)
print(f"\nAverage chunk length : {avg_len:.0f} words")
print(f"Test query           : '{QUERY}'")
print("\nModule 1 complete. Run Module 2 next.")


KNOWLEDGE BASE LOADED
  Doc 0 |  14 words | [FAQ    ] 01_FD_FAQ.pdf
         Premature withdrawal of FD incurs a 1 percent penalty on the appl...
  Doc 1 |  15 words | [POLICY ] 04_FD_Policy_Reference.pdf
         Fixed Deposit premature closure is allowed subject to applicable ...
  Doc 2 |  76 words | [SOP    ] 03_FD_SOPs.pdf
         This SOP covers Fixed Deposit premature withdrawal procedures. St...
  Doc 3 |  13 words | [PRODUCT] 02_FD_Product_Guide.pdf
         The Fixed Deposit interest rate for 24 months is 7.1 percent per ...
  Doc 4 |  13 words | [PRODUCT] 02_FD_Product_Guide.pdf
         Senior citizens receive an additional 0.5 percent interest on all...
  Doc 5 |  13 words | [POLICY ] 04_FD_Policy_Reference.pdf
         Early exit from your deposit account will attract a deduction fro...

Average chunk length : 24 words
Test query           : 'premature withdrawal penalty FD'

Module 1 complete. Run Module 2 next.


### Module 2: Binary Keyword Search

This is what `classify_by_keyword_rules()` from Chapter 1 does, applied to retrieval — just a yes/no check, no ranking.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Binary Keyword Search (Chapter 1 style)
#
# The rule engine from Chapter 1 only asks: does this word exist in
# this document, yes or no? It has no idea about ranking or scoring.
# We recreate that exact logic here, but applied to our knowledge base
# instead of a single email, so we can compare it against TF-IDF/BM25.
# ------------------------------------------------------------------

def binary_keyword_search(query: str, corpus_texts: list) -> list:
    """
    Returns 1 if ANY query word appears in the document, else 0.
    No frequency counting. No ranking. Purely yes/no per document.
    """
    query_words = set(tokenize(query))
    results = []
    for text in corpus_texts:
        doc_words = set(tokenize(text))
        # set intersection -- true if the doc shares at least 1 word with query
        match = 1 if query_words & doc_words else 0
        results.append(match)
    return results


matches = binary_keyword_search(QUERY, CORPUS_TEXTS)

print("=" * 65)
print("BINARY KEYWORD SEARCH (Chapter 1 rule-engine style)")
print("=" * 65)
print(f"Query: '{QUERY}'\n")
for i, (match, label) in enumerate(zip(matches, LABELS)):
    tag = "MATCH" if match else "no match"
    print(f"  Doc {i} [{tag:>8}] {label}")

print("\nProblem: every matching document gets the SAME score (1).")
print("There is no way to know which matching doc is MORE relevant.")
print("That ranking ability is exactly what TF-IDF and BM25 add next.")
print("\nModule 2 complete. Run Module 3 next.")


BINARY KEYWORD SEARCH (Chapter 1 rule-engine style)
Query: 'premature withdrawal penalty FD'

  Doc 0 [   MATCH] [FAQ    ] 01_FD_FAQ.pdf
  Doc 1 [   MATCH] [POLICY ] 04_FD_Policy_Reference.pdf
  Doc 2 [   MATCH] [SOP    ] 03_FD_SOPs.pdf
  Doc 3 [no match] [PRODUCT] 02_FD_Product_Guide.pdf
  Doc 4 [no match] [PRODUCT] 02_FD_Product_Guide.pdf
  Doc 5 [no match] [POLICY ] 04_FD_Policy_Reference.pdf

Problem: every matching document gets the SAME score (1).
There is no way to know which matching doc is MORE relevant.
That ranking ability is exactly what TF-IDF and BM25 add next.

Module 2 complete. Run Module 3 next.


### Module 3: TF-IDF Retrieval

Adds real ranking using word rarity (IDF) and word frequency (TF). Also shows where it breaks (repetition problem).

In [3]:

# ------------------------------------------------------------------
# MODULE 3: TF-IDF Retrieval
#
# Unlike Module 2, this gives every document an actual numeric score,
# so we can rank them from most to least relevant.
#
# We will also see TF-IDF's repetition problem: Doc 2 (the long, repetitive
# SOP) may score unfairly high just because it repeats words many times.
# ------------------------------------------------------------------

# TfidfVectorizer builds the TF-IDF matrix for us: one row per document,
# one column per unique word in the whole corpus
vectorizer = TfidfVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None)

# fit_transform: learns the vocabulary AND converts every doc to a TF-IDF vector
tfidf_matrix = vectorizer.fit_transform(CORPUS_TEXTS)

# transform the query using the SAME vocabulary learned above --
# this is important, the query must live in the same vector space as the docs
query_vector = vectorizer.transform([QUERY])

# cosine similarity between the query vector and every document vector
# gives us a score between 0 (nothing alike) and 1 (identical direction)
tfidf_scores = sklearn_cosine(query_vector, tfidf_matrix).flatten()

# sort documents by score, highest first
ranked_indices = np.argsort(tfidf_scores)[::-1]

print("=" * 65)
print("TF-IDF RETRIEVAL (ranked by cosine similarity)")
print("=" * 65)
print(f"Query: '{QUERY}'\n")
for rank, idx in enumerate(ranked_indices, start=1):
    print(f"  Rank {rank} | score={tfidf_scores[idx]:.4f} | {LABELS[idx]}")
    print(f"           {CORPUS_TEXTS[idx][:65]}...")

print("\nWatch Doc 2 (the repetitive SOP) -- notice its score is pulled up")
print("by repeating 'premature withdrawal' many times, not because it is")
print("more relevant than the short, clean FAQ answer (Doc 0).")
print("This is the repetition problem BM25 fixes with the k1 parameter.")
print("\nModule 3 complete. Run Module 4 next.")


TF-IDF RETRIEVAL (ranked by cosine similarity)
Query: 'premature withdrawal penalty FD'

  Rank 1 | score=0.7127 | [SOP    ] 03_FD_SOPs.pdf
           This SOP covers Fixed Deposit premature withdrawal procedures. St...
  Rank 2 | score=0.5159 | [FAQ    ] 01_FD_FAQ.pdf
           Premature withdrawal of FD incurs a 1 percent penalty on the appl...
  Rank 3 | score=0.1871 | [POLICY ] 04_FD_Policy_Reference.pdf
           Fixed Deposit premature closure is allowed subject to applicable ...
  Rank 4 | score=0.0000 | [POLICY ] 04_FD_Policy_Reference.pdf
           Early exit from your deposit account will attract a deduction fro...
  Rank 5 | score=0.0000 | [PRODUCT] 02_FD_Product_Guide.pdf
           Senior citizens receive an additional 0.5 percent interest on all...
  Rank 6 | score=0.0000 | [PRODUCT] 02_FD_Product_Guide.pdf
           The Fixed Deposit interest rate for 24 months is 7.1 percent per ...

Watch Doc 2 (the repetitive SOP) -- notice its score is pulled up
by repeating 'pre

### Module 4: BM25 Retrieval

Same ranking idea as TF-IDF, but with the two BM25 fixes: `k1` (repetition saturation) and `b` (length normalization).

In [4]:

# ------------------------------------------------------------------
# MODULE 4: BM25 Retrieval
#
# BM25Okapi from the rank_bm25 library does the same job as TF-IDF above,
# but with two extra parameters that fix TF-IDF's weaknesses:
#   k1 -> controls how fast repeated words stop adding to the score
#   b  -> controls how much long documents get penalised vs short ones
# ------------------------------------------------------------------

# tokenize every document once -- BM25Okapi needs a list of token lists,
# not raw strings
tokenized_corpus = [tokenize(text) for text in CORPUS_TEXTS]

# build the BM25 index -- this precomputes IDF for every word in the corpus
# k1=1.2 and b=0.75 are the standard defaults used in most search engines
bm25 = BM25Okapi(tokenized_corpus, k1=1.2, b=0.75)

# tokenize the query the same way we tokenized the documents
query_tokens = tokenize(QUERY)

# get_scores returns one BM25 score per document, already comparable/rankable
bm25_scores = bm25.get_scores(query_tokens)

# sort documents by score, highest first
ranked_indices = np.argsort(bm25_scores)[::-1]

print("=" * 65)
print("BM25 RETRIEVAL (k1=1.2, b=0.75)")
print("=" * 65)
print(f"Query: '{QUERY}'\n")
for rank, idx in enumerate(ranked_indices, start=1):
    print(f"  Rank {rank} | score={bm25_scores[idx]:.4f} | {LABELS[idx]}")
    print(f"           {CORPUS_TEXTS[idx][:65]}...")

print("\nCompare this ranking to Module 3 (TF-IDF).")
print("Doc 0 (short, clean FAQ) should now rank closer to the top,")
print("because BM25's k1 saturation stops Doc 2 from winning purely")
print("by repeating words many times.")
print("\nModule 4 complete. Run Module 5 next.")


BM25 RETRIEVAL (k1=1.2, b=0.75)
Query: 'premature withdrawal penalty FD'

  Rank 1 | score=1.7758 | [SOP    ] 03_FD_SOPs.pdf
           This SOP covers Fixed Deposit premature withdrawal procedures. St...
  Rank 2 | score=1.4171 | [FAQ    ] 01_FD_FAQ.pdf
           Premature withdrawal of FD incurs a 1 percent penalty on the appl...
  Rank 3 | score=0.0000 | [PRODUCT] 02_FD_Product_Guide.pdf
           Senior citizens receive an additional 0.5 percent interest on all...
  Rank 4 | score=0.0000 | [POLICY ] 04_FD_Policy_Reference.pdf
           Early exit from your deposit account will attract a deduction fro...
  Rank 5 | score=0.0000 | [PRODUCT] 02_FD_Product_Guide.pdf
           The Fixed Deposit interest rate for 24 months is 7.1 percent per ...
  Rank 6 | score=0.0000 | [POLICY ] 04_FD_Policy_Reference.pdf
           Fixed Deposit premature closure is allowed subject to applicable ...

Compare this ranking to Module 3 (TF-IDF).
Doc 0 (short, clean FAQ) should now rank closer to the 

### Module 5: k1 Saturation — Seeing the Diminishing-Returns Curve

Proves the claim from the theory: going from TF=1 to TF=40 barely increases the score once k1 saturation kicks in.

In [5]:

# ------------------------------------------------------------------
# MODULE 5: k1 Saturation Curve
#
# This isolates JUST the term-frequency part of the BM25 formula so we
# can see the "diminishing returns" behaviour directly, without any
# other document mixed in.
#
# Formula piece we are testing:
#   saturation(tf) = (tf * (k1 + 1)) / (tf + k1)
# ------------------------------------------------------------------

def bm25_tf_saturation(tf: int, k1: float = 1.2) -> float:
    """
    Returns the BM25 term-frequency saturation value for a given
    raw term frequency (tf). Higher tf should give diminishing,
    not linear, increases in score.
    """
    return (tf * (k1 + 1)) / (tf + k1)


print("=" * 65)
print("BM25 k1 SATURATION -- term frequency vs saturated score")
print("=" * 65)
print(f"{'Raw TF':>8} | {'Saturated Score (k1=1.2)':>26}")
print("-" * 40)

test_frequencies = [1, 2, 5, 10, 20, 40]
for tf in test_frequencies:
    score = bm25_tf_saturation(tf, k1=1.2)
    print(f"{tf:>8} | {score:>26.3f}")

score_at_1 = bm25_tf_saturation(1)
score_at_40 = bm25_tf_saturation(40)
percent_increase = ((score_at_40 - score_at_1) / score_at_1) * 100

print(f"\nGoing from TF=1 to TF=40 (a 40x increase in repetition)")
print(f"only increases the saturated score by {percent_increase:.0f}%.")
print("This is the diminishing-returns curve that stops repetition")
print("from dominating the final BM25 score.")
print("\nModule 5 complete. Run Module 6 next.")


BM25 k1 SATURATION -- term frequency vs saturated score
  Raw TF |   Saturated Score (k1=1.2)
----------------------------------------
       1 |                      1.000
       2 |                      1.375
       5 |                      1.774
      10 |                      1.964
      20 |                      2.075
      40 |                      2.136

Going from TF=1 to TF=40 (a 40x increase in repetition)
only increases the saturated score by 114%.
This is the diminishing-returns curve that stops repetition
from dominating the final BM25 score.

Module 5 complete. Run Module 6 next.


### Module 6: IDF Comparison — TF-IDF vs BM25

Shows both formulas side by side on the same corpus so you can see how their IDF values differ.

In [6]:

# ------------------------------------------------------------------
# MODULE 6: IDF Comparison -- plain TF-IDF style vs BM25 style
#
# Both formulas boost rare words and suppress common words, but the
# exact math is slightly different. BM25's IDF comes from a probability
# model, which is why it can even go negative for very common words
# in a very small corpus (a known edge case).
# ------------------------------------------------------------------

def plain_idf(term: str, tokenized_corpus: list) -> float:
    """Standard TF-IDF style IDF: log(N / document_frequency)."""
    N = len(tokenized_corpus)
    df = sum(1 for doc in tokenized_corpus if term in doc)
    if df == 0:
        return 0.0
    return math.log(N / df)


def bm25_idf(term: str, tokenized_corpus: list) -> float:
    """BM25 style IDF: log((N - df + 0.5) / (df + 0.5) + 1)."""
    N = len(tokenized_corpus)
    df = sum(1 for doc in tokenized_corpus if term in doc)
    return math.log((N - df + 0.5) / (df + 0.5) + 1)


# words chosen to show a spread: very common, medium, and rare in our corpus
test_words = ["fd", "premature", "withdrawal", "penalty", "senior"]

print("=" * 65)
print("IDF COMPARISON: plain TF-IDF style vs BM25 style")
print("=" * 65)
print(f"{'Word':<12} | {'Doc Freq':>8} | {'Plain IDF':>10} | {'BM25 IDF':>10}")
print("-" * 50)

for word in test_words:
    df = sum(1 for doc in tokenized_corpus if word in doc)
    p_idf = plain_idf(word, tokenized_corpus)
    b_idf = bm25_idf(word, tokenized_corpus)
    print(f"{word:<12} | {df:>8} | {p_idf:>10.3f} | {b_idf:>10.3f}")

print("\nNote: with a very small corpus like this 6-doc demo, if a word")
print("appears in MORE than half the documents, BM25's IDF formula can")
print("even go negative -- a known BM25 edge case on tiny corpora.")
print("This does not happen in the plain TF-IDF formula.")
print("\nModule 6 complete. Run Module 7 next.")


IDF COMPARISON: plain TF-IDF style vs BM25 style
Word         | Doc Freq |  Plain IDF |   BM25 IDF
--------------------------------------------------
fd           |        2 |      1.099 |      1.030
premature    |        3 |      0.693 |      0.693
withdrawal   |        2 |      1.099 |      1.030
penalty      |        3 |      0.693 |      0.693
senior       |        1 |      1.792 |      1.540

Note: with a very small corpus like this 6-doc demo, if a word
appears in MORE than half the documents, BM25's IDF formula can
even go negative -- a known BM25 edge case on tiny corpora.
This does not happen in the plain TF-IDF formula.

Module 6 complete. Run Module 7 next.


### Module 7: Hinglish Vocabulary Mismatch

The one weakness BM25 (and TF-IDF) cannot fix — zero word overlap means a score of exactly zero, even when the meaning matches.

In [7]:

# ------------------------------------------------------------------
# MODULE 7: Hinglish Vocabulary Mismatch
#
# 64% of the real project's emails are Hinglish. Policy documents are
# in English. If a customer's query shares ZERO words with a policy
# chunk, BM25 gives that chunk a score of exactly 0 -- no partial credit,
# even if the meaning is identical.
#
# We simulate that here with English policy chunks and a few Hinglish
# queries that mean the same thing.
# ------------------------------------------------------------------

# English-only policy chunks (like the real project's knowledge base)
ENGLISH_CHUNKS = [
    "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",
    "At maturity the FD principal and interest are credited to your registered bank account.",
    "Fixed Deposit nomination facility is available for all account holders.",
]

# each tuple is (description, query text)
HINGLISH_QUERIES = [
    ("English query (works fine)",
     "premature withdrawal FD penalty"),
    ("Hinglish query -- same meaning, zero word overlap",
     "FD band karna hai penalty kitni hogi"),
    ("Hinglish maturity query",
     "machurity ka paisa nahi aaya mera account mein"),
    ("Manual synonym expansion (like fd_keyword_groups.txt does)",
     "premature withdrawal nikalna penalty FD band karna"),
]

tokenized_chunks = [tokenize(c) for c in ENGLISH_CHUNKS]
bm25_hinglish = BM25Okapi(tokenized_chunks, k1=1.2, b=0.75)

print("=" * 65)
print("HINGLISH VOCABULARY MISMATCH -- BM25's biggest weakness")
print("=" * 65)
print("\nEnglish policy chunks:")
for i, chunk in enumerate(ENGLISH_CHUNKS):
    print(f"  Chunk {i}: {chunk[:65]}...")

for label, query in HINGLISH_QUERIES:
    scores = bm25_hinglish.get_scores(tokenize(query))
    print(f"\nQuery [{label}]")
    print(f"  '{query}'")
    print(f"  BM25 scores: {[round(s, 3) for s in scores]}")
    if all(s == 0.0 for s in scores):
        print("  -> ALL SCORES ARE ZERO. No shared words with any chunk.")
    else:
        best_idx = int(np.argmax(scores))
        print(f"  -> Best match: Chunk {best_idx} (score {max(scores):.3f})")

print("\nWhat this proves:")
print("  1. A pure Hinglish query can score exactly 0 against every")
print("     relevant document, even though the MEANING matches perfectly.")
print("  2. Manually adding English synonyms to the query (query expansion)")
print("     recovers some score -- but this does not scale to every possible")
print("     phrasing a customer might use.")
print("  3. This is exactly why Dense Retrieval (Topic 2) and Hybrid Search")
print("     (Topic 4) exist -- they catch meaning-based matches that BM25")
print("     structurally cannot.")

print("\n" + "=" * 65)
print("CHAPTER 7 TOPIC 1 -- KEY NUMBERS TO REMEMBER")
print("=" * 65)
print("""
  Binary keyword search -> yes/no only, no ranking
  TF-IDF  -> ranks documents, but repeated words inflate score forever
  BM25    -> k1 caps the benefit of repetition, b corrects for doc length
  BM25 score = 0 whenever query and document share zero words

  k1=1.2 default: TF=1 -> 1.0, TF=40 -> ~2.14 (about 2x, not 40x)
  b=0.75 default: partially corrects for document length differences

  Biggest real-world risk for this project: 64% Hinglish emails vs
  English policy documents -> vocabulary mismatch is the dominant
  failure mode, not a rare edge case.

  Next: Topic 2 (Dense Retrieval) solves vocabulary mismatch using meaning
        Topic 4 (Hybrid Search + RRF) combines BM25 and Dense together
""")
print("Module 7 complete. All Topic 1 modules done.")


HINGLISH VOCABULARY MISMATCH -- BM25's biggest weakness

English policy chunks:
  Chunk 0: Premature withdrawal of FD incurs a 1 percent penalty on the appl...
  Chunk 1: At maturity the FD principal and interest are credited to your re...
  Chunk 2: Fixed Deposit nomination facility is available for all account ho...

Query [English query (works fine)]
  'premature withdrawal FD penalty'
  BM25 scores: [np.float64(1.571), np.float64(0.101), np.float64(0.0)]
  -> Best match: Chunk 0 (score 1.571)

Query [Hinglish query -- same meaning, zero word overlap]
  'FD band karna hai penalty kitni hogi'
  BM25 scores: [np.float64(0.591), np.float64(0.101), np.float64(0.0)]
  -> Best match: Chunk 0 (score 0.591)

Query [Hinglish maturity query]
  'machurity ka paisa nahi aaya mera account mein'
  BM25 scores: [np.float64(0.0), np.float64(0.0), np.float64(0.559)]
  -> Best match: Chunk 2 (score 0.559)

Query [Manual synonym expansion (like fd_keyword_groups.txt does)]
  'premature withdrawal nika